In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Configurar la gestión de ruido con Estimator

<Accordion>
<AccordionItem title="Versiones de paquetes">

El código de esta página fue desarrollado con los siguientes requisitos.
Recomendamos usar estas versiones o más recientes.

```
qiskit-ibm-runtime~=0.46.1
```
</AccordionItem>
</Accordion>
Hay varias formas de gestionar el ruido, generalmente usando diversas técnicas de mitigación y supresión de errores para evitar errores antes de que ocurran. Estas técnicas suelen generar una sobrecarga de preprocesamiento. Por ello, es importante lograr un equilibrio entre perfeccionar los resultados y garantizar que el trabajo se complete en un tiempo razonable.

Estimator admite las siguientes técnicas de gestión de ruido. Consulta [Técnicas de mitigación y supresión de errores](error-mitigation-and-suppression-techniques) para ver una explicación de cada una. Consulta la sección [Configuración de error personalizada](#advanced-error) para ver instrucciones sobre cómo habilitar estas técnicas.

- [desacoplamiento dinámico](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-dynamical-decoupling-options#dynamicaldecouplingoptions)
- [Pauli twirling](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-twirling-options)
- [PEA](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-zne-options)
- [PEC](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-resilience-options-v2#pec)
- [`resilience_level`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-estimator-options#resilience_level)
- [TREX](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-resilience-options-v2#measure_mitigation)
- [ZNE](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-resilience-options-v2#zne)

<span id="resilience"></span>
## Nivel de resiliencia
El `resilience_level` especifica cuánta resiliencia se debe construir frente a los errores. Los niveles más altos generan resultados más precisos, a costa de tiempos de procesamiento más largos. Los niveles de resiliencia se pueden usar para configurar el equilibrio costo/precisión al aplicar la gestión de ruido a tu consulta primitiva. La gestión de ruido reduce los errores (sesgo) en los resultados procesando las salidas de una colección, o conjunto, de circuits relacionados. El grado de reducción de errores depende del método aplicado. El nivel de resiliencia abstrae la elección detallada del método de gestión de ruido para permitir a los usuarios razonar sobre el equilibrio costo/precisión que es apropiado para su aplicación.

Dado esto, cada nivel corresponde a un método o métodos con nivel creciente de sobrecarga de muestreo cuántico para permitirte experimentar con diferentes compensaciones tiempo-precisión. La siguiente tabla muestra qué niveles y métodos correspondientes están disponibles para cada uno de los primitivos.

<span id="resilience-table"></span>

| Nivel de resiliencia | Descripción | Técnica |
|------------------|-------------------------------------------------------------------------------------------------------|---------------------------------------------------------------------------|
| 0                | Sin mitigación | Ninguna |
| 1 [Predeterminado] | Costos mínimos de mitigación: Mitiga el error asociado con errores de lectura | Twirling de medición [Twirled Readout Error eXtinction (TREX)](/guides/error-mitigation-and-suppression-techniques#trex) |
| 2                | Costos de mitigación medios. Típicamente reduce el sesgo en los estimadores, pero no está garantizado que sea de sesgo cero. | Nivel 1 + [Zero Noise Extrapolation (ZNE)](/guides/error-mitigation-and-suppression-techniques#zne) y gate twirling

> **Info:** Los niveles de resiliencia se encuentran actualmente en beta, por lo que la sobrecarga de muestreo y la calidad de la solución variarán de circuit a circuit. Las nuevas funciones, opciones avanzadas y herramientas de gestión se lanzarán de forma continua. No se garantiza que se apliquen métodos específicos de gestión de ruido en cada nivel de resiliencia.

### Configurar Estimator con niveles de resiliencia
Puedes usar los niveles de resiliencia para especificar técnicas de gestión de ruido, o puedes establecer técnicas personalizadas individualmente como se describe en [Configuración de error personalizada](#advanced-error).

> **Note:** Cualquier opción que especifiques manualmente además del nivel de resiliencia se aplica en adición al conjunto base de opciones definido por el nivel de resiliencia. Por lo tanto, en principio, podrías establecer el nivel de resiliencia en 1, pero luego desactivar la mitigación de medición, aunque esto no es aconsejable.
> 
> Por ejemplo, establecer el nivel de resiliencia en 0 desactiva `zne_mitigation`, pero `estimator.options.resilience.zne_mitigation = True` anula ese valor.

### Ejemplo
El siguiente código habilita ZNE, TREX y gate twirling estableciendo `resilience_level 2`.

In [1]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorV2 as Estimator

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

# Setting options during primitive initialization
estimator = Estimator(backend, options={"resilience_level": 2})

<span id="advanced-error"></span>
## Configuración personalizada de gestión de ruido
Puedes activar y desactivar métodos individuales de gestión de ruido usando las [opciones de Estimator](/guides/estimator-options).

> **Note:** No todas las opciones funcionan juntas en todos los tipos de circuits. Consulta la [tabla de compatibilidad de funciones](estimator-options#options-compatibility-table) para más detalles.

### Ejemplo

In [2]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorV2 as Estimator

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

estimator = Estimator(backend)
options = estimator.options
# Turn on gate twirling.
options.twirling.enable_gates = True
# Turn on measurement error mitigation.
options.resilience.measure_mitigation = True

print(
    f">>> gate twirling is turned on: {estimator.options.twirling.enable_gates}"
)
print(
    f">>> measurement error mitigation is turned on: {estimator.options.resilience.measure_mitigation}"
)

>>> gate twirling is turned on: True
>>> measurement error mitigation is turned on: True


## Turn off all error mitigation

For instructions to turn off all error mitigation see the [Turn off all error suppression and mitigation](estimator-options#no-error-mitigation) section in the Estimator options guide.

## Next steps

<Admonition type="tip" title="Recommendations">
  - Walk through an example that uses error mitigation in the [Cost function lesson](/learning/courses/variational-algorithm-design/cost-functions) in IBM Quantum Learning.
  - Learn more about [error mitigation and error suppression techniques](error-mitigation-and-suppression-techniques).
  - Explore [Estimator options](/docs/guides/estimator-options).
  - Decide what [execution mode](/docs/guides/execution-modes) to run your job in.
</Admonition>